# Gold Layer - Modelo Dimensional (Star Schema)

Construye el modelo dimensional a partir de `e_commerce.silver.ventas_limpias` y lo persiste en el esquema `e_commerce.gold`.

## Tablas generadas

* `dim_cliente` — clientes únicos (nombre, email) con `id_cliente` como surrogate key
* `dim_ubicacion` — combinaciones únicas ciudad/dirección con `id_ubicacion`
* `dim_producto` — catálogo de productos con `id_producto`
* `dim_estado_pago` — estados de pago normalizados con `id_estado_pago`
* `dim_tiempo` — calendario derivado (año, mes, día, trimestre, día de la semana) con `id_tiempo`
* `fact_ventas` — tabla de hechos que une cada transacción con las surrogate keys de todas las dimensiones

## Procesos clave

* **Parseo de fechas**: múltiples formatos soportados vía `try_to_date`, más expresiones relativas (`Hoy`, `Ayer`, `Hace 3 dias`, `Semana pasada`)
* **Surrogate keys**: `xxhash64` sobre los atributos naturales de cada dimensión
* **Carga**: escritura Delta en modo `overwrite` hacia `e_commerce.gold`

In [0]:
from pyspark.sql.functions import (
    col, xxhash64, year, month, dayofmonth, quarter, dayofweek,
    coalesce, expr, when, current_date, date_sub,
)

In [0]:
df_silver = (
    spark.read.table("e_commerce.silver.ventas_limpias")
    .withColumn(
        "fecha",
        coalesce(
            expr("try_to_date(fecha_operacion, 'yyyy-MM-dd HH:mm:ss')"),
            expr("try_to_date(fecha_operacion, 'yyyy-MM-dd')"),
            expr("try_to_date(fecha_operacion, 'MM/dd/yyyy')"),
            expr("try_to_date(fecha_operacion, 'dd/MM/yyyy')"),
            expr("try_to_date(lower(fecha_operacion), 'd-MMM-yyyy')"),
            when(col("fecha_operacion") == "Hoy", current_date()),
            when(col("fecha_operacion") == "Ayer", date_sub(current_date(), 1)),
            when(col("fecha_operacion") == "Hace 3 dias", date_sub(current_date(), 3)),
            when(col("fecha_operacion") == "Semana pasada", date_sub(current_date(), 7)),
        ),
    )
)

In [0]:
df_dim_ubicacion = (
    df_silver.select("ciudad", "direccion_envio")
    .distinct()
    .withColumn("id_ubicacion", xxhash64(col("ciudad"), col("direccion_envio")))
)

In [0]:
df_dim_cliente = (
    df_silver.select("nombre_cliente", "email")
    .distinct()
    .withColumn("id_cliente", xxhash64(col("nombre_cliente"), col("email")))
)

In [0]:
df_dim_producto = (
    df_silver.select("nombre_producto")
    .distinct()
    .withColumn("id_producto", xxhash64(col("nombre_producto")))
)

In [0]:
df_dim_pago = (
    df_silver.select("estado_pago")
    .distinct()
    .withColumn("id_estado_pago", xxhash64(col("estado_pago")))
)

In [0]:
df_dim_tiempo = (
    df_silver.select("fecha")
    .distinct()
    .filter(col("fecha").isNotNull())
    .withColumn("anio", year(col("fecha")))
    .withColumn("mes", month(col("fecha")))
    .withColumn("dia", dayofmonth(col("fecha")))
    .withColumn("trimestre", quarter(col("fecha")))
    .withColumn("dia_semana", dayofweek(col("fecha")))
    .withColumn("id_tiempo", xxhash64(col("fecha")))
)

In [0]:
df_fact_ventas = (
    df_silver.alias("s")
    .join(df_dim_cliente.alias("dc"), (col("s.nombre_cliente") == col("dc.nombre_cliente")) & (col("s.email") == col("dc.email")), "left")
    .join(df_dim_ubicacion.alias("du"), (col("s.ciudad") == col("du.ciudad")) & (col("s.direccion_envio") == col("du.direccion_envio")), "left")
    .join(df_dim_producto.alias("dp"), col("s.nombre_producto") == col("dp.nombre_producto"), "left")
    .join(df_dim_pago.alias("dpago"), col("s.estado_pago") == col("dpago.estado_pago"), "left")
    .join(df_dim_tiempo.alias("dt"), col("s.fecha") == col("dt.fecha"), "left")
    .select(
        col("s.id_transaccion"),
        col("dc.id_cliente"),
        col("du.id_ubicacion"),
        col("dp.id_producto"),
        col("dpago.id_estado_pago"),
        col("dt.id_tiempo"),
        col("s.cantidad_producto"),
        col("s.monto_total")
    )
)

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS e_commerce.gold")

DataFrame[]

In [0]:
df_dim_cliente.write.format("delta").mode("overwrite").saveAsTable("e_commerce.gold.dim_cliente")
df_dim_ubicacion.write.format("delta").mode("overwrite").saveAsTable("e_commerce.gold.dim_ubicacion")
df_dim_producto.write.format("delta").mode("overwrite").saveAsTable("e_commerce.gold.dim_producto")
df_dim_pago.write.format("delta").mode("overwrite").saveAsTable("e_commerce.gold.dim_estado_pago")
df_dim_tiempo.write.format("delta").mode("overwrite").saveAsTable("e_commerce.gold.dim_tiempo")
df_fact_ventas.write.format("delta").mode("overwrite").saveAsTable("e_commerce.gold.fact_ventas")